# Post5G - Simple Scenario

## Prerequisites
* You have successfully followed the [Preparation](./preparation.ipynb), [Provisioning Resources](./provision.ipynb), and [Kubernetes Cluster](./k8s.ipynb) tutorials.
* You have basic Python knowledge
* You have a good understanding of the 5G architecture

## Experiment for this scenario

This scenario aims to introduce to the orchestrator by deploying the following 5G network.

![Simple topology](figures/simple.png)


It is composed of a 5G core implemented with `Open5GS`, one gNB implemented with `OAI` called *gnb*, and two UE implemented with `OAI` too, called *sim_ue1* and *sim_ue2*.

We only have one DNN (`internet`) and one S-NSSAI (`internet`).

The UEs are configured as follows:

| UE | SUPI | S-NSSAI | DNN |
| --- | --- | --- | --- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `internet` | `internet` |

We will deploy all the core functions on a single machine, called `node-core`;

We will deploy the gNB on a single machine, called `node-ran`;

We will deploy all the UEs on a single machine, called `node-ues`;

The connection between the UEs and the gNB is simulated and there is no need for a specific radio unit.

As the network is fully virtual (the core and RAN is emulated while the radio channel and UEs are simulated), we can deploy everything on virtual machines.

To that aim, we will use ephemeral virtual machines on the [SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html) but one could use their own virual machines or even baremetal machines.



## Metamodel and Deployement model definition

### Metamodel
The Post-5G blueprint abstracts the 5G network with a generic **metamodel**
compliant with 3GPP specifications. The metamodel is essentially a
[Pydantic model](https://pydantic.dev/docs/validation/dev/get-started/) that
helps to define the settings of the 5G network to operate in a declarative way.

To simplify the usage of the metamodel, it comes with a full set of default
values. That way, the user only needs to specify the settings for elements that
deviate from the default values.

In this example we follow a leaning-by-doing approach, but you can visit the
[metamodel website](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/meta-model)
to read the full specifications and default values.

In our case, we have to define a core, one gNB, and two UEs as stated above.

In this example, we will deploy a core with the default settings so we don't have
to explicitly mention and configure it.

We however have to declare the gNB and the UEs as described above. To that aim,
we simply have to instanciate an object of the `MetaModel` class and explicitely
populate it with our desired elements.

The following snippet defines the desired RAN and UEs.


In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
ran:
    components:
        gnb:
            type: gnb
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
""")

**NOTE:** as the radio channel is simulated we explicitly list the gNBs to which
the UEs are connected.

### Deployment model

The particularity of the metamodel is that it is implementation agnostic. This
allows one to share their 5G information and experiment in various environments
and implementations.

Our orchestrator library is used to instantiate metamodels to these specific
environments. To do so, it complements the metamodel with a **deployment model**
that carries information about a specific deployment of the metamodel.

It is in the deployment model that we specify the implementation of the different
5G functions but also where and how to deploy them.

Similarly to the metamodel, the deployment model is a Pydantic model with default
values to simplify using it.

To create the deployment model we just have to create an instance of the
`DeploymentModel` class and populate it accordingly to our needs.

By default the core is in Open5GS with our orestrator so we don't have to
specify anything. However, we have to explicitly specify that the gNB and the UEs
use OAI. Hence, their explicit presence in the deployment model under the
`ran` and `ues` sections of the deployment model.

We also have to explicitly announce the nodes on which the functions must be
deployed. For that, we have to list theses machines and tag them with the roles
that can support in the `nodes` section of the deployment model. We allow the
`node-core` machine to run the core and the RIC with the `core` and `ric` tags.
With its only `ran` tag, the `core-ran` machine only accepts RAN components and
with the single `ue` tag, the `core-ues` machine only accepts UE instances.

We said that we wanted to use ephemeral virtual machines in the SLICES BI
infrastructure. For this exact purpose, the orchestrator offers a collection of
provisionners that can be used to provision machines. In our case, the
provisioner called `my-provisionner` is used and configured to automatically
spawn and configure virtual machines in the SLICES BI. This is why each machine
that we defined has the `provisioner: my-provisionner` attribute.

Each provisioner has its own set of configuration parameters. In this example,
we decide to use the `fr-sophia2-bi-vm1` SLICES-BI infrastructure and the
`post5g-beta` project and `simple_scenario` experiment. We ask the provisioner
to automatically create the experiment for us and nodes will have a lifetime of
8h after which they will be automatically destroyed.

**IMPORTANT NOTE:** make sure your public keys are registered to SLICES before
deploying your network.

You can go back to the [preparation tutorial](./preparation.ipynb) if you are
not familiar yet with these fundamental SLICES-RI concepts.



**NOTE:** components are referenced by name in the metamodel and the deployment
model. Therefore, the `provisioner: my-provisionner` attribute in a node means:
use the provisionner named `my-provisionner` to provision the resource. This
means that if you want to use another type of provisioner, you don't have to
change the nodes definition but only the provisionner definition. This name based
referencing is particularly for exploring space of parameters.

**NOTE:** by changing the provisionner type, one can change the way the resources
will be deployed (e.g., deploy bare metal machines, onboard already running
Linux hosts...).

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  components:
    gnb:
      type: gnb
      implementation: oai
ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized
nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: cnria2026
      experiment: simple_scenario_backup
      create_experiment: true
      experiment_duration: 24h
      node_duration: 24h
"""
)



## Provisioning the infrastructure

### Create ephemeral virtual machines

Now the we have defined the metamodel and deployment model, we can trigger the
provisioning of the resources with the adequate **Manager**. Managers are design
for specific tasks only, for example, the `ProvisioningManagement` role is to
provision the machines found in the `nodes` section of the deployment
model. It automatically determines which exact provisionner to use for each node.

First, we create a provisioning manager by specifying the metamodel and
deployment model it must use. This stage has no effect on the infrastructure, it
just reconciles the models and validate their correction.

Then we can trigger the actual provisioning of the nodes.

In [3]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'cnria2026' for 'node-ues' provisioning] ***

TASK [provisioning/bi/provision : Select project 'cnria2026' for 'n

We now have our three virtual machines deployed in the SLICES BI and configured
according to the needs of this scenario. 

*Optional:* You can verify it by listing the resources of the experiment with
the SLICES CLI (adapt the parameters to your setup).

In [27]:
!SLICES_BI_SITE_ID=fr-sophia2-bi-vm1 slices bi list-resources --experiment simple_scenario_backup

  Resources in experiment exp_expauth.ilabt.imec.be_01m4ydz6g0e76vw0wgc7hmejfk  
┏━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃           ┃           ┃        ┃ Created   ┃ Expires  ┃ Public    ┃ Private  ┃
┃ ID        ┃ Name      ┃ Status ┃ At        ┃ At       ┃ IPv4      ┃ IPv4     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ r_fr-sop… │ node-ues  │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │        │ 00:04 UTC │ 00:04    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_fr-sop… │ node-core │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │        │ 00:04 UTC │ 00:04    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_fr-sop… │ node-ran  │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │   

Each manager consumes a metamodel and a deployment model and  the result of its
actions is the creation of a new deployment model that can be used by the manager
next in line.

Many different methods exist to obtain the metamodel and/or  deployment model
of a manager. For example the `output()` method directly returns the two models.
As you expected, the deployment model contains all the information to acces the
freshly created machines.

In [5]:
metamodel, deployment_model = prov.output()
print (deployment_model.to_yaml())

provisioners:
  my-provisionner:
    type: bi
    params:
      project: cnria2026
      experiment: simple_scenario_backup
      create_experiment: true
      node_duration: 24h
nodes:
  node-core:
    is_provisioned: true
    provisioner: my-provisionner
    ssh_info:
      hostname: 10.123.92.110
      user: dsaucezi
      jumphost:
        hostname: bastion1.theblueprintfactory.org
        user: proxy
        common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
          ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o
          ControlPersist=7200s -o ControlPath=/tmp/cm-%C
      common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
        ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s
        -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no
        -o UserKnownHostsFile=/dev/null  -W %h:%p proxy@bastion1.theblueprintfactory.org
        -p 

It can also be handy to save the models to files, for example to use them later
on (we make a case for it later in this tutorial).

In [26]:
from pathlib import Path
DIR="state/simple"
Path(DIR).mkdir(parents=True, exist_ok=True)

prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

### Build kubernetes cluster

The orchestrator relies on kubernetes to deploy most of its functions. The next
step is then to create a kubernetes cluster composed of the virtual machines we
have deployed in the previous step.

To do so, we can use the `K8SManagement` manager, which role is to deploy and
configure a kubernetes cluster compatible with the 5G network deployement that
we want. In this case, it will be composed of the `node-core`, `node-ran`, and
`node-ues` machines, labeled in a way that the workload is scheduled correctly
on them (e.g., UEs can only be deployed on `node-ues`).

As mentionned earlier, managers consume metamodel and deployment model. Here we
want to use the output of the provisioning we did in the previous step. When we
instantiate our kubernetes manager, we just have to mention which manager
preceeds it (in this case `prov`) and it will automatically retrieve its models.

**IMPORTANT:** our orchestrator library follows a declarative and functional
approach, as such, the metamodel and deployment model are copied when passed
from one manager to another meaning that the input models are never modified by
the manager.

Once the manager is properly instantiated, we trigger the deployment of the
cluster on the machines. It will create a kubernetes cluster with all the nodes
and install the corresponding `kubeconfig` file on all the machines to simplify
later operations.

**NOTE:** to onboard existing kubernetes clusters please refer to the
[orchestrator](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator)
library documentation.

In [7]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)            # don't show output

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [8]:
k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core.

The task is as simple as instantiating a `CoreManagement` Manager and deploying
it as all information about the 5G specific settings and the deployment details
have already been defined in the metamodel and the deployment model. As for the
other managers, we can feed it with the models from the manager of the previous
stage (i.e., `k8s`).

The effect of the deployment is to instantiate all the 5G core functions but
also to register in the UDR the information about the two UEs described in the
metamodel.

In [9]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-smf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node-co

By default the Open5GS core is deployed in the `open5gs-core` namespace in the
cluster. Functions of the core are deployed only on the nodes with the tagged
role `core`. To verify that the core is deployed properly, we can list all the
pods of that namespace.

The `ProvisioningManagement` manager offers the simple yet powerful `run_node_command`
function to execute bash commands on nodes. In this case, we can run the
`kubectl get pods` command on the `node-core` machine to check that pods are
deployed as expected.

**NOTE:** remember that the orchestrator library follows a functional approach
meaning that the metamodel and deployment model of managers are not altered by
the execution of other managers. This is why we have to make a new
instance of `ProvisioningManagement` manager and import models from the core
deployement manager we just executed. Indeed, the deployement model of the
former `ProvisioningManagement` instance does not correspond to the latest one.

In [10]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True))["node-core"])

NAME                            READY   STATUS    RESTARTS   AGE     IP               NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-6glkc    1/1     Running   0          2m37s   10.244.134.7     node-core   <none>           <none>
open5gs-ausf-5778d7548b-b28mr   1/1     Running   0          2m19s   10.244.134.8     node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-9k685    1/1     Running   0          24s     10.244.134.14    node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          3m30s   10.244.134.5     node-core   <none>           <none>
open5gs-nrf-65d98494d7-kdrsn    1/1     Running   0          3m1s    10.244.134.6     node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-49tw8   1/1     Running   0          40s     10.244.134.13    node-core   <none>           <none>
open5gs-pcf-6f444b66c5-qvrkr    1/1     Running   0          102s    10.244.134.10    node-core   <none>           <none>
open5gs-smf-5c7

In a very similar way, the logs of 5G functions can be retrieved directly with
the managers. For example, the logs of the AMF shows that it is well registered
to the NRF.

In [11]:
print(await core.get_logs(components=["amf"], quiet=True))

Open5GS daemon v2.8.0

10/15 00:17:28.260: [app] INFO: Configuration: '/opt/open5gs/etc/open5gs/config.yaml' (../lib/app/ogs-init.c:144)
10/15 00:17:28.260: [app] INFO: File Logging: '/opt/open5gs/var/log/open5gs/amf.log' (../lib/app/ogs-init.c:147)
10/15 00:17:28.260: [app] INFO: LOG-LEVEL: 'info' (../lib/app/ogs-init.c:150)
10/15 00:17:28.263: [sbi] INFO: Setup NF EndPoint(fqdn) [open5gs-nrf:0] (../lib/sbi/context.c:451)
10/15 00:17:28.263: [metrics] INFO: metrics_server() [http://10.244.134.7]:9090 (../lib/metrics/prometheus/context.c:392)
10/15 00:17:28.263: [sbi] INFO: NF Service [namf-comm] (../lib/sbi/context.c:2012)
10/15 00:17:28.263: [sbi] INFO: nghttp2_server() [http://10.244.134.7]:80 (../lib/sbi/nghttp2-server.c:454)
10/15 00:17:28.286: [amf] INFO: ngap_server() [10.244.134.7]:38412 (../src/amf/ngap-sctp.c:61)
10/15 00:17:28.286: [sctp] INFO: AMF initialize...done (../src/amf/app.c:33)
10/15 00:17:28.292: [sbi] INFO: [ce221fa2-c82d-41f1-bdc4-650a81042aa1] NF registered [He

### Deploy the 5G RAN

Now that we have a fully functional 5G core, we can deploy our RAN by using a
`RANManagement` manager. We instanciate such a managet and import the metamodel
and deployment model from the manager of our last deployement stage (i.e.,
`core`) then deploy the RAN by colling the `deploy` function of the manager.


In [12]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node-core]

TASK [helm/prepare : Render templated Helm files for 'oai-gnb'] ****************
changed: [node-core] => (item=values.yaml)
changed: [node-core] => (item=config.yaml)

PLAY RECAP *********************************************************************
node-core                  : ok=4

In our scenario, we defined the RAN as being one OAI gNB name `gnb`, by
default the namespace for OAI RAN is `oai-ran`. After the deployment,
there should be one pod and it should be deployed in the `node-ran` node.

As for the core, we can create an instance of `ProvisioningManagement` and list
pods in that namespace by running the `kubectl` command.

In [24]:
prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True))["node-ran"])

NAME                       READY   STATUS    RESTARTS   AGE     IP               NODE       NOMINATED NODE   READINESS GATES
oai-gnb-777f6d8567-nx4qp   1/1     Running   0          7m57s   10.244.164.193   node-ran   <none>           <none>


The output of the command confirms the proper deployment of the gNB on the
`node-ran` machine.

To confirm that the gNB is implemented with OAI and is registered to the
core, we can analyse the logs of the gNB.

In [14]:
print(await ran.get_logs(components=["gnb"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 621312
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1

From the logs, we can confirm that the NG Setup procedure was successful with
the AMF and by combining this logs with the listing of the pods in the core, we
can see that the address of the AMF used by the gNB correspond to the address of
the AMF of the core we deployed earlier. This IP address has been setup
transparently by our orchestrator library, removing the burden for users.

## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the UEs we have
defined in our metamodel.

For that we can instantiate a `UEManagement` manager and feed it with the 
metamodel and deployment model obtained from the manager used in the previous
stage, then deploy the UEs by calling the `deploy` function of that manager.

When implemented with OAI, UEs are deployed in the `oai-ue` namespace
by default. We can also use the `kubectl` command to validate that the two UEs
have been deployed.

In [15]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue1'] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.co

In [25]:
prov = ProvisioningManagement(preceding=ues)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True))["node-ues"])

NAME                           READY   STATUS    RESTARTS   AGE     IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue1-77fdccc764-szsd2   1/1     Running   0          5m30s   10.244.198.3   node-ues   <none>           <none>
oai-sim-ue2-6644849f47-hdc8j   1/1     Running   0          5m31s   10.244.198.2   node-ues   <none>           <none>


As for the gNB or the AMF we can also analyse the logs the the UEs and see that
they both successuflly registered to the network and established a PDU session.

In [17]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

In [18]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [20]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True)

ping from `sim_ue1`:

In [21]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
[1792024242.420117] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=37.4 ms
[1792024243.416098] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=31.9 ms
[1792024244.419440] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=35.2 ms
[1792024245.418629] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=33.1 ms
[1792024246.423962] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=37.7 ms
[1792024247.420185] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=32.2 ms
[1792024248.429305] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=40.0 ms
[1792024249.433124] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=42.7 ms
[1792024250.432476] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=40.3 ms
[1792024251.434827] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=41.5 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9011ms
rtt min/avg/max/mdev = 31.947/37.198/42.725/3.744 ms



ping from `sim_ue2`:

In [22]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1792024242.541094] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=23.8 ms
[1792024243.548473] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=29.3 ms
[1792024244.538733] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=18.4 ms
[1792024245.559352] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=38.1 ms
[1792024246.553091] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=30.8 ms
[1792024247.543930] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=20.7 ms
[1792024248.559114] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=34.8 ms
[1792024249.552719] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=26.6 ms
[1792024250.568493] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=41.2 ms
[1792024251.562767] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=34.4 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9011ms
rtt min/avg/max/mdev = 18.356/29.796/41.220/7.121 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [23]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")